In [15]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Шаг 1. Создание легковесной выборки (Turbo Dataset)

Уменьшает объем данных для быстрого отбора фичей.

In [16]:
!python 00_create_small_dataset.py

🚀 Creating Turbo Dataset (Smart Sampling)...
Original shape: (750000, 42)
Found 24 rare targets (<1% frequency).
Rare clients: 60210 (Keep 100%)
Common clients: 689790 (Sample 20.0%)
✅ Final Turbo Dataset: 198168 rows (26.4% of original)
Processing Main features...
🎉 Turbo Dataset created in ./data_turbo
You can now train on this small dataset extremely fast!


## Шаг 2. Отбор признаков (Feature Selection)

Извлекает самые важные признаки через LightGBM. Запустите оба скрипта (один для мета-признаков, второй для финальных моделей).



In [17]:
!python 01a_extract_top_feat.py   # Генерирует selected_features_gain_700.pkl
!python 01b_select_importance.py  # Генерирует selected_features_gain.pkl

Traceback (most recent call last):
  File "/Users/pavelkorkodinov/Hacks/DataFusionContest/cyber-shelf/solutions/super-puper-ensemble-0842/01a_extract_top_feat.py", line 4, in <module>
    import lightgbm as lgb
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/__init__.py", line 11, in <module>
    from .basic import Booster, Dataset, Sequence, register_logger
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/basic.py", line 9, in <module>
    from .libpath import _LIB  # isort: skip
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/libpath.py", line 49, in <module>
    _LIB = ctypes.cdll.LoadLibrary(_find_lib_path()[0])
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/ctypes/__init__.py", line 452, in LoadLibrary
    return self._dlltype(name)
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/ctypes/__init__.py", line 374, in __init__
    self._handle = _dlopen(self._name, mode)
OSError: dlopen(/

## Шаг 3. Генерация OOF Meta-признаков

Обучает базовые модели на 5 фолдах и собирает вероятности смежных классов.

In [11]:
!python 02_build_meta.py

Traceback (most recent call last):
  File "/Users/pavelkorkodinov/Hacks/DataFusionContest/cyber-shelf/solutions/super-puper-ensemble-0842/02_build_meta.py", line 6, in <module>
    import lightgbm as lgb
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/__init__.py", line 11, in <module>
    from .basic import Booster, Dataset, Sequence, register_logger
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/basic.py", line 9, in <module>
    from .libpath import _LIB  # isort: skip
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/libpath.py", line 49, in <module>
    _LIB = ctypes.cdll.LoadLibrary(_find_lib_path()[0])
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/ctypes/__init__.py", line 452, in LoadLibrary
    return self._dlltype(name)
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/ctypes/__init__.py", line 374, in __init__
    self._handle = _dlopen(self._name, mode)
OSError: dlopen(/opt/min

## Шаг 4. Расчет глобальных агрегаций

Считает быстрые статистики Polars по всем дополнительным признакам.

In [12]:
!python 03_make_global_agg.py

🚀 [1/2] Calculating Global Aggs for Train (Lazy Execution)...
/Users/pavelkorkodinov/Hacks/DataFusionContest/cyber-shelf/solutions/super-puper-ensemble-0842/03_make_global_agg.py:11: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = [c for c in train_lazy.columns if c != "customer_id"]
✅ Saved 'train_global_aggs.parquet' (Shape: (750000, 3))

🚀 [2/2] Calculating Global Aggs for Test (Lazy Execution)...
✅ Saved 'test_global_aggs.parquet' (Shape: (250000, 3))


## Шаг 5. Обучение финальных моделей

Обучение с сохранением чекпоинтов (при прерывании скрипт продолжит с последнего таргета).

In [13]:
!python 04a_train_meta.py         # Генерирует submission_META_SMART.parquet
!python 04b_train_global_agg.py   # Генерирует submission_CHAMPION_AGGS_SEED.parquet

Traceback (most recent call last):
  File "/Users/pavelkorkodinov/Hacks/DataFusionContest/cyber-shelf/solutions/super-puper-ensemble-0842/04a_train_meta.py", line 1, in <module>
    import lightgbm as lgb
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/__init__.py", line 11, in <module>
    from .basic import Booster, Dataset, Sequence, register_logger
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/basic.py", line 9, in <module>
    from .libpath import _LIB  # isort: skip
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/lightgbm/libpath.py", line 49, in <module>
    _LIB = ctypes.cdll.LoadLibrary(_find_lib_path()[0])
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/ctypes/__init__.py", line 452, in LoadLibrary
    return self._dlltype(name)
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/ctypes/__init__.py", line 374, in __init__
    self._handle = _dlopen(self._name, mode)
OSError: dlopen(/opt/mi

## Шаг 6. Финальный блендинг

Смешивает предсказания двух пайплайнов через ранговое усреднение и приводит типы к float64 для проверяющей системы.

In [14]:
!python 05_rank_blend.py

Traceback (most recent call last):
  File "/Users/pavelkorkodinov/Hacks/DataFusionContest/cyber-shelf/solutions/super-puper-ensemble-0842/05_rank_blend.py", line 8, in <module>
    df_old = pd.read_parquet(SUB_OLD)
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/pandas/io/parquet.py", line 669, in read_parquet
    return impl.read(
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/pandas/io/parquet.py", line 258, in read
    path_or_handle, handles, filesystem = _get_path_or_handle(
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/pandas/io/parquet.py", line 141, in _get_path_or_handle
    handles = get_handle(
  File "/opt/miniconda3/envs/hackathon/lib/python3.10/site-packages/pandas/io/common.py", line 882, in get_handle
    handle = open(handle, ioargs.mode)
FileNotFoundError: [Errno 2] No such file or directory: 'submission_META_SMART.parquet'
